# 00 · Test connessione BigQuery

Verifica dell'ambiente: elenco dataset/tabelle, metadati delle tabelle principali e lettura dell'ultima notizia disponibile.

Regole di costo (vedi `CLAUDE.md`): niente `SELECT *`, filtro sempre sulla colonna di partizione, dry run prima di ogni query, conferma esplicita sopra 1 GB.

## a. Import e client

In [ ]:
import warnings

import pandas as pd
from google.cloud import bigquery

# L'avviso sul "quota project" delle credenziali utente è innocuo: il progetto è passato esplicitamente
warnings.filterwarnings("ignore", message=".*quota project.*")

PROJECT = "class-hackaton-12"
MAX_BYTES_SENZA_CONFERMA = 1024**3  # 1 GB

client = bigquery.Client(project=PROJECT)


def human_bytes(n):
    n = float(n or 0)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:,.1f} {unit}"
        n /= 1024
    return f"{n:,.1f} PB"


def dry_run(sql, params=None):
    """Stima i byte che la query leggerebbe, senza eseguirla (gratuito)."""
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False, query_parameters=params or [])
    job = client.query(sql, job_config=cfg)
    print(f"Dry run: {job.total_bytes_processed:,} byte stimati ({human_bytes(job.total_bytes_processed)})")
    return job.total_bytes_processed


def run_query(sql, params=None, confermato=False):
    """Dry run + esecuzione. Oltre 1 GB si ferma finché non si passa confermato=True."""
    stima = dry_run(sql, params)
    if stima > MAX_BYTES_SENZA_CONFERMA and not confermato:
        raise RuntimeError(f"Stima {human_bytes(stima)} > 1 GB: serve conferma esplicita (confermato=True).")
    cfg = bigquery.QueryJobConfig(
        query_parameters=params or [],
        maximum_bytes_billed=max(stima * 2, 10 * 1024**2),  # rete di sicurezza
    )
    job = client.query(sql, job_config=cfg)
    df = job.to_dataframe(create_bqstorage_client=False)
    print(f"Eseguita: {job.total_bytes_billed or 0:,} byte fatturati ({human_bytes(job.total_bytes_billed)})")
    return df


print("Client pronto sul progetto:", client.project)

## b. Dataset e tabelle

In [ ]:
for ds in client.list_datasets():
    print(f"{ds.dataset_id}")
    for t in client.list_tables(ds.dataset_id):
        print(f"  └─ {t.table_id} ({t.table_type})")

## c. Metadati delle tabelle (gratuiti)

In [ ]:
TABELLE = [
    "news.articles",
    "financial_instruments.instruments_quotes",
    "financial_instruments.instruments_embeddings",
]

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)

for tid in TABELLE:
    t = client.get_table(f"{PROJECT}.{tid}")
    tp = t.time_partitioning
    print(f"=== {tid} ===")
    print(f"Righe:            {t.num_rows:,}")
    print(f"Dimensione:       {human_bytes(t.num_bytes)} ({t.num_bytes:,} byte)")
    if tp:
        print(f"Partizione:       colonna={tp.field or '_PARTITIONTIME'}, tipo={tp.type_}")
    elif t.range_partitioning:
        print(f"Partizione:       range su {t.range_partitioning.field}")
    else:
        print("Partizione:       nessuna")
    print(f"Clustering:       {t.clustering_fields}")
    schema = pd.DataFrame(
        [(f.name, f.field_type, f.mode, f.description or "") for f in t.schema],
        columns=["colonna", "tipo", "mode", "descrizione"],
    )
    display(schema)
    print()

## d. Partizione più recente di `news.articles`

Prima si prova `INFORMATION_SCHEMA.PARTITIONS` (legge solo metadati). Se non è accessibile, si cerca `MAX(data_modifica)` negli ultimi 7 giorni e poi negli ultimi 30, fermandosi appena si trovano dati.

In [ ]:
sql_partitions = f"""
SELECT partition_id, total_rows
FROM `{PROJECT}.news.INFORMATION_SCHEMA.PARTITIONS`
WHERE table_name = 'articles'
  AND partition_id NOT IN ('__NULL__', '__UNPARTITIONED__')
  AND total_rows > 0
ORDER BY partition_id DESC
LIMIT 1
"""

ultima_data = None
try:
    df_part = run_query(sql_partitions)
    if not df_part.empty:
        ultima_data = pd.to_datetime(df_part.loc[0, "partition_id"], format="%Y%m%d").date()
        print(f"Via INFORMATION_SCHEMA: partizione {df_part.loc[0, 'partition_id']} con {df_part.loc[0, 'total_rows']:,} righe")
except Exception as e:
    print(f"INFORMATION_SCHEMA.PARTITIONS non accessibile: {e}")

if ultima_data is None:
    for giorni in (7, 30):
        sql_max = f"""
        SELECT MAX(data_modifica) AS ultima
        FROM `{PROJECT}.news.articles`
        WHERE data_modifica >= DATETIME_SUB(CURRENT_DATETIME(), INTERVAL {giorni} DAY)
        """
        df_max = run_query(sql_max)
        if pd.notna(df_max.loc[0, "ultima"]):
            ultima_data = df_max.loc[0, "ultima"].date()
            print(f"Trovati dati negli ultimi {giorni} giorni")
            break
        print(f"Nessun dato negli ultimi {giorni} giorni")

print("Partizione più recente:", ultima_data)

## e. Dry run della query per l'ultima notizia

In [ ]:
assert ultima_data is not None, "Nessuna partizione recente trovata"

# Solo la partizione più recente (filtro su data_modifica) e solo le colonne necessarie
sql_ultima = f"""
SELECT data_pubblicazione, data_modifica, titolo, sommario, body
FROM `{PROJECT}.news.articles`
WHERE data_modifica >= DATETIME(@giorno)
  AND data_modifica <  DATETIME_ADD(DATETIME(@giorno), INTERVAL 1 DAY)
ORDER BY data_modifica DESC
LIMIT 1
"""
params_ultima = [bigquery.ScalarQueryParameter("giorno", "DATE", ultima_data)]

stima = dry_run(sql_ultima, params_ultima)
print("Sotto la soglia di 1 GB:", stima <= MAX_BYTES_SENZA_CONFERMA)

## f. Ultima notizia

In [ ]:
df_news = run_query(sql_ultima, params_ultima)
n = df_news.iloc[0]
testo = n["body"] or n["sommario"] or ""

print("Data pubblicazione:", n["data_pubblicazione"])
print("Data modifica:     ", n["data_modifica"])
print("Titolo:            ", n["titolo"])
print()
print(testo[:500])